# 07 — Универсальный Kaggle submission

Этот notebook восстанавливает **одного уже измеренного кандидата**, обучает его на всём train и создаёт проверенный inference-файл. Он не выполняет CV и не меняет champion.


## Правила

1. Выбор делается устойчивым Candidate ID, а не свободной комбинацией признаков и estimator.
2. EXP со статусом reject разрешён, если его feature hook проходит inference audit.
3. Модель из CV не переиспользуется: pipeline заново fit на всём train.
4. Запись CSV, model artifact и SUB-карточки выполняется только последней ячейкой.


In [1]:
from pathlib import Path
import importlib
import sys

from IPython.display import display

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = next(
    candidate for candidate in (CURRENT_DIR, *CURRENT_DIR.parents)
    if (candidate / 'README.md').exists() and (candidate / 'src').exists()
)
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from ml_project import DataCatalog
import ml_project.config as project_config
import ml_project.baseline_config as baseline_config
import ml_project.model_screening_config as screening_config
import ml_project.modeling as modeling_tools
import ml_project.submission as submission_tools

print(f'Корень проекта: {PROJECT_ROOT}')


Корень проекта: D:\Projects\Titanik-ML-project


## 1. Каталог доступных кандидатов

В таблицу автоматически попадают baseline, сохранённые EXP-кандидаты и все модели из MS-screening. Статус reject не скрывает кандидата.


In [2]:
project_config = importlib.reload(project_config)
baseline_config = importlib.reload(baseline_config)
screening_config = importlib.reload(screening_config)
submission_tools = importlib.reload(submission_tools)

initial_settings = baseline_config.BASELINE
MODEL_GROUPS = screening_config.MODEL_GROUPS
catalog = submission_tools.candidate_catalog(
    PROJECT_ROOT,
    baseline_model_label=modeling_tools.resolved_model_name(initial_settings),
)
display(submission_tools.candidate_catalog_report(catalog))


,Candidate ID,Источник,Feature set,Модель,Метрика,Статус,CV mean ± std
0,EXP-001/model,baseline,EXP-001,logistic_regression,NaN,baseline,—
1,EXP-002/candidate,experiment,EXP-002,logistic_regression,accuracy,adopt,0.8036 ± 0.0163
2,EXP-003/candidate,experiment,EXP-003,logistic_regression,accuracy,adopt,0.8204 ± 0.0176
3,EXP-004/candidate,experiment,EXP-004,logistic_regression,accuracy,reject,0.8160 ± 0.0158
4,EXP-005/candidate,experiment,EXP-005,logistic_regression,accuracy,reject,0.8137 ± 0.0132
5,EXP-006/candidate,experiment,EXP-006,logistic_regression,accuracy,reject,0.8193 ± 0.0154
6,EXP-007/candidate,experiment,EXP-007,logistic_regression,accuracy,reject,0.8182 ± 0.0196
7,EXP-008/candidate,experiment,EXP-008,logistic_regression,accuracy,reject,0.8182 ± 0.0212
8,EXP-009/candidate,experiment,EXP-009,experiment candidate,NaN,reject,—
9,EXP-010/candidate,experiment,EXP-010,logistic_regression,accuracy,reject,0.8148 ± 0.0162


## 2. Выбор submission

Редактируйте только следующую ячейку. Для нового файла всегда используйте новый `SUB-ID`.


In [3]:
SUBMISSION_ID = "SUB-006"
SELECTED_CANDIDATE = "MS-003/gradient_boosting"

selected = catalog[catalog['candidate_id'].eq(SELECTED_CANDIDATE)]
if selected.empty:
    raise KeyError(f'Кандидат не найден: {SELECTED_CANDIDATE}')
display(submission_tools.candidate_catalog_report(selected))


,Candidate ID,Источник,Feature set,Модель,Метрика,Статус,CV mean ± std
21,MS-003/gradient_boosting,screening,EXP-013,gradient_boosting,accuracy,shortlisted,0.8372 ± 0.0223


## 3. Данные и версии

Sample submission используется, если файл настроен и существует. Без него notebook построит двухколоночный файл из KEY и TARGET.


In [4]:
DATASETS = project_config.DATASETS
FEATURE_GROUPS = project_config.FEATURE_GROUPS
KEY = project_config.KEY
RAW_DIR = project_config.RAW_DIR
TARGET = project_config.TARGET
TRAIN_DATASET = project_config.TRAIN_DATASET
INFERENCE_DATASET = project_config.INFERENCE_DATASET

files = DataCatalog(PROJECT_ROOT, RAW_DIR, DATASETS)
files.validate()
train = files.load(TRAIN_DATASET)
inference = files.load(INFERENCE_DATASET)
available = set(files.available_names())
sample_dataset = next(
    (
        name for name, spec in DATASETS.items()
        if spec.get('role') == 'submission_example' and name in available
    ),
    None,
)
sample_submission = files.load(sample_dataset) if sample_dataset else None
file_report = files.file_report().set_index('dataset')
dataset_versions = {
    'train': str(file_report.loc[TRAIN_DATASET, 'sha256']),
    'inference': str(file_report.loc[INFERENCE_DATASET, 'sha256']),
}
print('Train:', train.shape)
print('Inference:', inference.shape)
print('Sample submission:', sample_dataset or 'не найден — будет создан по контракту')


Train: (891, 12)
Inference: (418, 11)
Sample submission: не найден — будет создан по контракту


## 4. Восстановление и inference audit

До fit notebook показывает точный source, feature set, estimator и состояние кода. Контекстно-зависимые raw-признаки будут перечислены в audit и заблокируют predict.


In [5]:
prepared = submission_tools.prepare_submission_candidate(
    PROJECT_ROOT,
    SELECTED_CANDIDATE,
    train,
    inference,
    FEATURE_GROUPS,
    target=TARGET,
    key=KEY,
    initial_settings=initial_settings,
    model_groups=MODEL_GROUPS,
)
display(submission_tools.submission_contract_report(prepared))
if prepared.inference_audit.empty:
    print('Inference audit: passed')
else:
    display(prepared.inference_audit)


,Поле,Значение
0,Candidate ID,MS-003/gradient_boosting
1,Источник,screening: MS-003
2,Feature set,EXP-013
3,Feature module,ml_project.experiments.exp_013_tt_comb
4,Модель,gradient_boosting → GradientBoostingClassifier
5,Исходное решение,shortlisted
6,CV,accuracy: 0.8372 ± 0.0223
7,Model features,10
8,Train rows,891
9,Inference rows,418


Inference audit: passed


## 5. Full-train fit и predict

Эта ячейка ещё ничего не сохраняет. Проверьте validation, распределение и первые/последние строки.


In [6]:
prediction = submission_tools.fit_submission_candidate(
    prepared,
    sample_submission,
    key=KEY,
    target=TARGET,
)
display(prediction.validation)
display(prediction.prediction_distribution)
display(prediction.frame.head())
display(prediction.frame.tail())


,check,observed,expected,passed
0,rows,418,418,True
1,unique key,418,418,True
2,key set,418,418,True
3,missing predictions,0,0,True


,prediction,rows,share
0,0,295,0.705742
1,1,123,0.294258


,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,0


,PassengerId,Survived
413,1305,0
414,1306,1
415,1307,0
416,1308,0
417,1309,1


## 6. Явно сохранить официальный SUB-run

После визуальной проверки выполните последнюю ячейку. CSV и fitted pipeline останутся локальными в `artifacts/submissions/<SUB-ID>/`; карточка и компактный registry — в `submissions/`.


In [7]:
saved = submission_tools.save_submission(
    PROJECT_ROOT,
    SUBMISSION_ID,
    prediction,
    dataset_versions=dataset_versions,
    allow_overwrite=True,
)
print('CSV для Kaggle:', saved.submission_path.relative_to(PROJECT_ROOT))
print('Fitted pipeline:', saved.model_path.relative_to(PROJECT_ROOT))
print('Metadata:', saved.metadata_path.relative_to(PROJECT_ROOT))
print('Карточка:', saved.note_path.relative_to(PROJECT_ROOT))
print('Registry:', saved.registry_path.relative_to(PROJECT_ROOT))


CSV для Kaggle: artifacts\submissions\SUB-006\SUB-006_MS-003_gradient_boosting.csv
Fitted pipeline: artifacts\submissions\SUB-006\SUB-006_model.joblib
Metadata: artifacts\submissions\SUB-006\metadata.json
Карточка: submissions\SUB-006.md
Registry: submissions\results.csv


## После загрузки на Kaggle

Откройте созданную `SUB-xxx` карточку, заполните поля `kaggle_public_score` / `kaggle_private_score` во frontmatter, дату и короткое наблюдение. Затем выполните следующую sync-ячейку: переобучение не требуется. Kaggle score — внешний сигнал, а не автоматическое решение о champion.


In [9]:
display(submission_tools.sync_submission_scores(PROJECT_ROOT))


,submission_id,candidate_id,kaggle_public_score,kaggle_private_score
0,SUB-001,MS-001/feature_champion,"""0.76315""","""0.76315"""
1,SUB-002,MS-001/random_forest,"""0.78708""","""0.78708"""
2,SUB-003,EXP-013/candidate,"""0.78468""","""0.78468"""
3,SUB-004,MS-002/random_forest,"""0.77990""","""0.77990"""
4,SUB-005,MS-003/hist_gradient_boosting,"""0.75358""","""0.75358"""
5,SUB-006,MS-003/gradient_boosting,"""0.76315""","""0.76315"""
